# 04 — Similarity Model (KNN / FAISS)

## Training
Fits a similarity index for nearest-neighbor retrieval and saves a new timestamped run folder under `artifacts/`.
- Output files (in `artifacts/<run_id>/`):
  - `similarity_index.joblib`
  - `preprocessors.joblib`
  - `case_table.csv` (rows used for display + retrofit voting)

**Backend**
- Small datasets: set `similarity.backend: sklearn`
- ~100k rows: `configs/colab_large.yaml` uses `faiss` (CPU) + SVD embedding

## Demo / Inference (after training)
You can query using the same JSON payload format as the severity notebook.
The output should include:
- `similar_cases` (nearest bridges)
- `recommended_retrofit` (vote from neighbors)

In [5]:
from google.colab import drive

drive.mount('/content/drive')

%cd /content/drive/MyDrive/retrofit

PROJECT_ROOT = '/content/drive/MyDrive/retrofit'
CONFIG = 'configs/colab_large.yaml'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/retrofit


In [6]:
!pip -q install -r requirements-colab.txt
!pip -q install -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for bridge-retrofit (pyproject.toml) ... done


In [7]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, '-m', 'bridge_retrofit.cli',
    '--config', CONFIG,
    '--project-root', PROJECT_ROOT,
    'fit-similarity',
])

0

In [9]:
import json
import subprocess
import sys
import pandas as pd

# Example JSON-like payload (missing keys are OK; extra keys are ignored)
example = {
    "_comment": "Provide any subset of feature columns; missing values are imputed.",
    "Bridge_Type": "Beam",
    "Material": "Steel",
    "Age_Years": 25,
    "Failure_Type": "Scour",
}

out = subprocess.check_output([
    sys.executable, '-m', 'bridge_retrofit.cli',
    '--config', CONFIG,
    '--project-root', PROJECT_ROOT,
    'predict',
    '--json', json.dumps(example),
], text=True)
print(out)

# Pretty-print the similarity part (if present)
result = json.loads(out)
print("Recommended retrofit:", result.get("recommended_retrofit"))
similar_cases = result.get("similar_cases") or []
if similar_cases:
    display(pd.DataFrame(similar_cases).head(10))
else:
    print("No similar cases returned (did you run the Training cell in this notebook?)")

{
  "severity_pred": "Severe",
  "severity_proba": [
    0.3279340670475811,
    0.3096465625285445,
    0.36241937042387445
  ],
  "similar_cases": [
    {
      "Bridge_Name": "MetroSpan_95713",
      "Failure_Type": "Bearing Failure",
      "Suggested_Retrofit": "Bearing replacement",
      "distance": 6.371624946594238
    },
    {
      "Bridge_Name": "RiverLink_43392",
      "Failure_Type": "Bearing Failure",
      "Suggested_Retrofit": "Bearing replacement",
      "distance": 14.049088478088379
    },
    {
      "Bridge_Name": "SkyBridge_29471",
      "Failure_Type": "Bearing Failure",
      "Suggested_Retrofit": "Bearing replacement",
      "distance": 15.08775520324707
    },
    {
      "Bridge_Name": "MetroSpan_53797",
      "Failure_Type": "Bearing Failure",
      "Suggested_Retrofit": "Bearing replacement",
      "distance": 15.752196311950684
    },
    {
      "Bridge_Name": "GangaSetu_43718",
      "Failure_Type": "Bearing Failure",
      "Suggested_Retrofit": "Bearing

,Bridge_Name,Failure_Type,Suggested_Retrofit,distance
0,MetroSpan_95713,Bearing Failure,Bearing replacement,6.371625
1,RiverLink_43392,Bearing Failure,Bearing replacement,14.049088
2,SkyBridge_29471,Bearing Failure,Bearing replacement,15.087755
3,MetroSpan_53797,Bearing Failure,Bearing replacement,15.752196
4,GangaSetu_43718,Bearing Failure,Bearing replacement,16.855799
